# System Dependencies
To get started with Unstructured.io, we need a few system-wide dependencies:

## Poppler (poppler-utils)
Handles PDF processing. It's a library that can extract text, images, and metadata from PDFs. Unstructured uses it to parse PDF documents and convert them into processable text.

## Tesseract (tesseract-ocr)
Optical Character Recognition (OCR) engine. When you have scanned documents, images with text, or PDFs that are essentially pictures, Tesseract reads the text from these images and converts it to machine-readable text.

## libmagic
File type detection library. It identifies what type of file you're dealing with (PDF, Word doc, image, etc.) by analyzing the file's content, not just the extension. This helps Unstructured choose the right processing method for each document.

In [ ]:
#%pip install -Uq "unstructured[all-docs]" 

Note: you may need to restart the kernel to use updated packages.


In [1]:
import json
from typing import List
import warnings
import os
import importlib
import torch
import tqdm as notebook_tqdm

# Unstructured for document parsing (existing library)
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from dotenv import load_dotenv

load_dotenv()

warnings.filterwarnings("ignore")

c:\Users\lovep\miniconda3\envs\ragApp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# --- Setup ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # Use GPU if available, else fallback to CPU
DB_PATH = os.getenv("VECTOR_DB_PATH", "db/chroma")  # Vector DB storage path, overridable via .env

print(f"Using {'GPU' if DEVICE == 'cuda' else 'CPU'} for compute.")

Using GPU for compute.


In [26]:
def partition_document(file_path: str) -> List[Document]:
    "Extract elements from PDF using unstructured."

    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="fast", # previously "hi_res" for high quality, but caused errors with ROCM; "fast" is quicker and good enough for most cases
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF (don't ignore images)
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )

    print(f"Extracted {len(elements)} elements from {file_path}.")
    return elements

file_path = "xarchiv.pdf" 
elements = partition_document(file_path)

list_elem = set([str(type(e)) for e in elements])  # Show the types of elements extracted
list_elem

No languages specified, defaulting to English.


Extracted 393 elements from xarchiv.pdf.


{"<class 'unstructured.documents.elements.Footer'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [21]:
elements[36].to_dict()  # Show the content of a specific element

{'type': 'NarrativeText',
 'element_id': '802f18c694bcba60958f2359fa9a9aa5',
 'text': 'Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position- wise fully connected feed-forward network. We employ a residual connection [11] around each of the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension dmodel = 512.',
 'metadata': {'coordinates': {'points': ((107.641, 502.58614439999997),
    (107.641, 579.0713408),
    (505.65748784059974, 579.0713408),
    (505.65748784059974, 502.58614439999997)),
   'system': 'PixelSpace',
   'layout_width': 612.0,
   'layout_height': 792.0},
  'filename':

In [22]:
images = [element for element in elements if element.category == "Image"]
print(f"Found {len(images)} images in the document.")

# There are images in the document, but due to dependencies in the unstructured library with ROCM, we arn't able to display them in this environment.
if len(images) > 0:
    print(f"First image element: {images[0].to_dict()}")

Found 0 images in the document.


In [ ]:
for element in list_elem:
    print(f"Element type: {element}")
    for e in elements:
        if str(type(e)) == element:
            print(f"  - Content:\n {e.to_dict()}")
            break  # Show only the first instance of each type

Element type: <class 'unstructured.documents.elements.NarrativeText'>
  - Content:
 {'type': 'NarrativeText', 'element_id': '364e32591a70248550ce7f70ee3ce068', 'text': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.', 'metadata': {'coordinates': {'points': ((124.313, 72.5901232), (124.313, 112.44032319999997), (487.8945424, 112.44032319999997), (487.8945424, 72.5901232)), 'system': 'PixelSpace', 'layout_width': 612.0, 'layout_height': 792.0}, 'filename': 'xarchiv.pdf', 'last_modified': '2026-08-06T01:52:22', 'page_number': 1, 'languages': ['eng'], 'filetype': 'application/pdf', 'parent_id': '435f4a1f9ccde8c962b310ac0457d565'}}
Element type: <class 'unstructured.documents.elements.Text'>
  - Content:
 {'type': 'UncategorizedText', 'element_id': '06dee51b8e61b52301595fc7bf93721b', 'text': '3 2 0 2', 'metadata': {'coordinates': {'points': ((16.34, 213.920000000000

In [29]:
def create_chunks_by_title(elements: List[Document]) -> List[Document]:
    "Chunk the document elements by title using unstructured's chunk_by_title."

    chunks = chunk_by_title(
        elements, 
        max_characters=3000,  # Maximum characters per chunk
        new_after_n_chars=2400,  # Start a new chunk after this many characters
        combine_text_under_n_chars=500,  # Combine text under this many characters into the previous chunk
    )
    print(f"Created {len(chunks)} chunks")
    return chunks

chunks = create_chunks_by_title(elements)
chunks

Created 33 chunks


In [30]:
chunks[2].to_dict()  # Show the content of the third chunk

{'type': 'CompositeElement',
 'element_id': '2fca3ed7-77c5-49db-a37a-f1b1c34cc40d',
 'text': 'Introduction\n\nRecurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been firmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation [35, 2, 5]. Numerous efforts have since continued to push the boundaries of recurrent language models and encoder-decoder architectures [38, 24, 15].\n\nRecurrent models typically factor computation along the symbol positions of the input and output sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently sequential nature precludes parallelization within training examples, which becomes critical at longer sequence lengths, as memory constraints limit batching across exam

In [ ]:
def separate_content_types(chunks: List[Document]) -> dict:
    "Separate chunks into text and image content types."

    text_chunks = [chunk for chunk in chunks if chunk.category == "Text"]
    image_chunks = [chunk for chunk in chunks if chunk.category == "Image"]

    print(f"Separated into {len(text_chunks)} text chunks and {len(image_chunks)} image chunks.")
    return {"text": text_chunks, "image": image_chunks}